![hslu_logo.png](./img/hslu_logo.png)


<hr style="border:1px solid black">

<h1 style="text-align:center;font-size:50px"><b>AAI - FS25</b></h1>
<p style="text-align:center;font-size:40px">Week 10</p>

---
---

**Remark**: Some text in this document was generated using Gemini.


# Table of contents for week 10
1. [Transformer](#transformer)
     1. [Attention](#attention)
     2. [Transformer Blocks](#transformer_blocks)
     3. [Parallelization](#parallelization)
     4. [Embeddings](#embeddings)
2. [LLM with Transformer](#llm_transformer)
     1. [Sampling](#sampling)
     2. [Self-Supervised Training](#training)
3. [Exercise](#exercise)
4. [Literature](#literature)


## 1. Transformer <a name="transformer"></a>
In the realm of artificial intelligence, particularly in deep learning, *transformers* refer to a type of neural network architecture that has revolutionized how machines process sequential data, especially language. The transformer architecture was proposed in the 2017 research paper "Attention Is All You Need" [[2]](#literature). In the following we look into this architecture. First, we go through an introduction of the main constituents of the transformer in [[1]](#literature). Second, in [Sect. 3](#exercise) we run some experiments with a basic implementation of the model.

### 1.1. Attention <a name="attention"></a>
The main concept behind the transformer architecture, *attention*, is explained in [[1, Chap. 9.1]](#literature).

### 1.2. Transformer Blocks <a name="transformer_blocks"></a>
In addition to attention, a transformer block includes other layers as explained in [[1, Chap. 9.2]](#literature).

### 1.3. Parallelization <a name="parallelization"></a>
A main advantage of the transformer is its ability to parallelize computation, as explained in [[1, Chap. 9.3]](#literature).

### 1.4. Embeddings <a name="embeddings"></a>
Word embedding, as well as potitional embedding, are explained in [[1, Chap. 9.4]](#literature).


## 2. LLM with Transformer <a name="llm_transformer"></a>
Most modern LLMs are based on the transformer architecture. They let the transformer work in an autoregressive fashion, that is, generating text by consecutively predicting the next word.

### 2.1. Sampling <a name="sampling"></a>
For the task of choosing a next word, also called *decoding* or *sampling*, there are different methods discussed in [[1, Chap. 10.2]](#literature):

- greedy decoding
- top-k sampling
- top-p sampling
- beam search (see [[1, Chap. 13.4]](#literature))

### 2.2. Self-Supervised Training <a name="training"></a>
LLMs are trained on very large corpora. The model is always given the correct history sequence to predict a next word. This principle is called *teacher forcing*. Find more information about training LLMs in [[1, Chap. 10.3]](#literature).


## 3. Exercise <a name="exercise"></a>
A transformer is implemented in the following Python code.
1. Try to understand the code for the model in the following cell from what we have learned in [Sect. 1](#transformer).


In [ ]:
# transformer model

import torch
from torch import nn, Tensor
import math

rec_attn_probs = None    # set to [] for recording rec_attn_probs

class MyTransformer(nn.Module):

    def __init__(self, n_token:int, d_model:int, n_heads:int, d_ff:int, n_layers:int, dropout:float=0.5):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(n_token, d_model)
        self.pos_encoder = MyPositionalEncoder(d_model)
        self.encoder_layers = nn.ModuleList([MyEncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.linear = nn.Linear(d_model, n_token)
        self.init_weights()
        self.att_mask = None

    def init_weights(self):
        initrange = 0.1
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.linear.bias.data.zero_()
        self.linear.weight.data.uniform_(-initrange, initrange)

    def forward(self, x:Tensor):
        # x: Tensor[batch_size, seq_len]
        # returns: Tensor[batch_size, seq_len, n_token]

        if (self.att_mask is None) or (self.att_mask.size(1)!=x.size(1)):
            self.att_mask = torch.triu(torch.ones(1, x.size(1), x.size(1)), diagonal=1).type_as(x).bool()
        # att_mask: Tensor[1, seq_len, seq_len]
        x = self.embedding(x) * math.sqrt(self.d_model)
        # x: Tensor[batch_size, seq_len, d_model]
        x = self.pos_encoder(x)
        for enc_layer in self.encoder_layers:
            x = enc_layer(x, self.att_mask)
        x = self.linear(x)
        return x

class MyPositionalEncoder(nn.Module):

    def __init__(self, d_model:int, max_len:int=5000):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0)/d_model))
        encoder_matrix = torch.zeros(max_len, d_model)
        encoder_matrix[:,0::2] = torch.sin(position * div_term)
        encoder_matrix[:,1::2] = torch.cos(position * div_term)
        encoder_matrix = encoder_matrix.unsqueeze(0)
        self.register_buffer("encoder_matrix", encoder_matrix, persistent=True)

    def forward(self, x:Tensor):
        # x: Tensor[batch_size, seq_len, d_model]
        x = x + self.encoder_matrix[:,:x.size(1),:]
        return x

class MyEncoderLayer(nn.Module):

    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        # x: Tensor[batch_size, seq_len, d_model]
        # mask: Tensor[seq_len, seq_len]
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

class PositionWiseFeedForward(nn.Module):

    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        global rec_attn_probs    # for recording attn_probs
        # Q, K, V: Tensor[batch_size, n_heads, seq_len, d_k]
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        # attn_scores: Tensor[batch_size, n_heads, seq_len, seq_len]
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask, -1e9)
        attn_probs = torch.softmax(attn_scores, dim=-1)
        # attn_probs: Tensor[batch_size, n_heads, seq_len, seq_len]
        if rec_attn_probs is not None:
            rec_attn_probs.append(attn_probs)    # record
        output = torch.matmul(attn_probs, V)
        # output: Tensor[batch_size, n_heads, seq_len, d_k]
        return output

    def split_heads(self, x):
        # x: Tensor[batch_size, seq_len, d_model]
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.n_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, Q, K, V, mask=None):
        # Q, K, V: Tensor[batch_size, seq_len, d_model]
        # mask: Tensor[1, seq_len, seq_len]
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        # Q, K, V: Tensor[batch_size, n_heads, seq_len, d_k]

        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        # attn_output: Tensor[batch_size, n_heads, seq_len, d_k]
        output = self.W_o(self.combine_heads(attn_output))
        # output: Tensor[batch_size, seq_len, d_model]
        return output

2. Read in some text samples along with a corresponding vocabulary. You may use the books of Karl May available on ILIAS. (They are extensive and also freely available, because very old. Other large corpora are welcome.)

In [ ]:
# read Karl May texts

import os

def read_data(path,files):
    text_data = ""
    for fle in files:
        with open(os.path.join(path,fle), encoding='utf-8', errors="ignore") as f:
            lines = f.readlines()
            text_data += ''.join(lines)
    return text_data

path = "./sample_data/KarlMay"
train_data_files = os.listdir(path)
train_data_files.remove("DieSklavenkarawane.txt")
val_data_files = ["DieSklavenkarawane.txt"]
print(train_data_files,val_data_files)

train_text_data = read_data(path,train_data_files)
val_text_data = read_data(path,val_data_files)


In [ ]:
import pickle

with open('vocabulary.pkl', 'rb') as f:
    token2id,id2token = pickle.load(f)

3. Prepare the data sets.

In [ ]:
# prepare train and val data

train_batch_size = 20
eval_batch_size = 10

def data_process(text_data):
    data = []
    for word in text_data.split():
        token = word.strip('><»«-–",!:;?„“‚‘†').strip("'").lower()
        if (len(token)>0):
            eos = (token[-1]=='.')
            token = token.strip('.')
            if (token not in token2id):
                token = '<unk>'
            data.append(torch.tensor([token2id[token]],dtype=torch.long))
            if eos:
                data.append(torch.tensor([token2id['<eos>']],dtype=torch.long))
    return torch.cat(tuple(data))

train_data = data_process(train_text_data)
val_data = data_process(val_text_data)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def batchify(data, batch_size):
    full_seq_len = data.size(0) // batch_size
    data = data[:full_seq_len*batch_size]
    data = data.view(batch_size,full_seq_len).contiguous()
    return data.to(device)

train_data = batchify(train_data, train_batch_size)
val_data = batchify(val_data, eval_batch_size)

print("training data set contains",train_data.shape,"tokens on device",device)
print("validation data set contains",val_data.shape,"tokens on device",device)

4. Now train the model. This may take quite a while on a CPU. With what loss function is the model trained?

In [ ]:
# create model and train

from tempfile import TemporaryDirectory

d_model = 200    # embedding dimension
d_ff = 200    # dimension of the feedforward network model in ``nn.TransformerEncoder``
n_layers = 6    # number of ``nn.TransformerEncoderLayer`` in ``nn.TransformerEncoder``
n_heads = 4    # number of heads in ``nn.MultiheadAttention``
dropout = 0.2    # dropout probability
n_tokens = len(token2id)    # size of vocabulary
model = MyTransformer(n_tokens, d_model, n_heads, d_ff, n_layers, dropout).to(device)

train_seq_len = 100

epochs = 1000

criterion = nn.CrossEntropyLoss()
lr = 5.0    # learning rate
optimizer = torch.optim.SGD(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 1.0, gamma=0.90)

def get_batch(source, i, batch_seq_len):
    # source: Tensor[batch_size, full_seq_len]
    # i: int, start position
    # batch_seq_len: int
    # returns: tuple(data, target), where data has shape [batch_size, seq_len], and target has shape [batch_size, seq_len]
    seq_len = min(batch_seq_len, source.size(1)-1-i)
    data = source[:,i:i+seq_len]
    targets = source[:,i+1:i+1+seq_len]
    return data, targets

def train(model, train_data, train_seq_len):
    model.train()    # turn on train mode
    total_loss = 0.0
    log_interval = 200
    num_batches = train_data.size(1) // train_seq_len

    for batch, i in enumerate(range(int(train_seq_len*torch.rand(1)), train_data.size(1)-1, train_seq_len)):
        data, targets = get_batch(train_data, i, train_seq_len)
        # data: Tensor, shape [batch_size, seq_len]
        # targets: Tensor, shape [batch_size, seq_len]
        output = model(data)
        # output: Tensor, of shape [batch_size, seq_len, n_tokens]
        output_flat = output.view(-1, n_tokens)
        # output_flat: Tensor, shape [batch_size*seq_len, n_tokens]
        loss = criterion(output_flat, targets.reshape(-1))

        optimizer.zero_grad()    # important to reset gradient
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()

        total_loss += loss.item()
        if ((batch % log_interval == 0) and (batch > 0)):
            lr = scheduler.get_last_lr()[0]
            loss = total_loss/log_interval
            ppl = math.exp(loss)
            print(f'| epoch {epoch:3d} | {batch:5d}/{num_batches:5d} batches | lr {lr:02.2f} | loss {loss:5.2f} | ppl {ppl:8.2f}')
            total_loss = 0.0

def evaluate(model, eval_data, train_seq_len):
    model.eval()    # turn on evaluation mode
    total_loss = 0.0
    with torch.no_grad():
        for i in range(0, eval_data.size(1)-1, train_seq_len):
            data, targets = get_batch(eval_data, i, train_seq_len)
            output = model(data)
            output_flat = output.view(-1, n_tokens)
            total_loss += criterion(output_flat, targets.reshape(-1)).item()
    return total_loss/((eval_data.size(1)-1)//train_seq_len)

for epoch in range(1, epochs+1):
    train(model, train_data, train_seq_len)
    val_loss = evaluate(model, val_data, train_seq_len)
    val_ppl = math.exp(val_loss)
    print('-' * 89)
    print(f'| end of epoch {epoch:3d} | valid loss {val_loss:5.2f} | valid ppl {val_ppl:8.2f}')
    print('-' * 89)
    scheduler.step()


5. Now sample and generate some sentences. (You may also use the following pretrained model available on ILIAS.)

In [ ]:
model = torch.load("KarlMay_model_6_4_250.pt", map_location=torch.device('cpu'), weights_only=False)

In [ ]:
# sample sentences

n_pred = 5
max_traces = 20
train_seq_len = 100
pred_len = 100
model.eval()

with torch.no_grad():
    s0 = int((val_data.shape[1]-train_seq_len)*torch.rand(1))
    generated_text = val_data[0,s0:s0+train_seq_len].view(1,-1)
    text_lp = torch.zeros((1))
    while (generated_text.shape[1]<train_seq_len+pred_len):
        next_lp = torch.empty((0),device=device)
        next_ii = torch.empty((0),dtype=int,device=device)
        for k in range(generated_text.shape[0]):
            prediction = model(generated_text[k,-train_seq_len:].view(1,-1))
            prediction[0,-1,0] = 0.0    # inhibit <unk>
            pp,ii = torch.sort(prediction[0,-1,:],descending=True)
            pp = pp[:n_pred]
            pp *= (0.7+0.6*torch.rand(n_pred,device=device))    # introduce randomness
            lp = pp+text_lp[k]
            next_lp = torch.cat((next_lp, lp))
            next_ii = torch.cat((next_ii, ii[:n_pred]))
        _,ii = torch.sort(next_lp,descending=True)
        next_generated = torch.empty((0,generated_text.shape[1]+1),dtype=int,device=device)
        for k in range(len(next_ii)):
            next_generated = torch.cat((next_generated,torch.cat((generated_text[ii[k]//n_pred,:],next_ii[ii[k]].view(1))).view(1,-1)),0)
        generated_text = next_generated[:min(next_generated.shape[0],max_traces),:]
        text_lp = next_lp[ii[:generated_text.shape[0]]]

ll_val = list(generated_text[0,train_seq_len-20:train_seq_len].detach().cpu().numpy())
print(" ".join([id2token[id] for id in ll_val]))
ll_pred = list(generated_text[0,train_seq_len:].detach().cpu().numpy())
print(" ".join([id2token[id] for id in ll_pred]))

6. What are your conclusions?

## 4. Literature <a name="literature"></a>

<a id="ref1" href="https://web.stanford.edu/~jurafsky/slp3">[1]</a> Daniel Jurafsky and James H. Martin. 2025. Speech and Language Processing: An Introduction to Natural Language Processing, Computational Linguistics, and Speech Recognition with Language Models, 3rd edition. Online manuscript released January 12, 2025.

<a id="ref2" href="https://web.stanford.edu/~jurafsky/slp3">[2]</a> Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin:
Attention Is All You Need. NIPS 2017.
